# 08 — Case Studies

Eğitilmiş `07_full_gnn_rich` modeli ile tarihsel önemli koalisyonların **v(S) trajectory**'sini yıl-yıl izler.

## Case study'ler
1. **NATO evolution** (1949-2016) — Kurucu 12'den 28'e
2. **AB evolution** (1957-2016) — AET-6'dan AB-28'e
3. **Varşova Paktı çöküşü** (1955-1991) — v(N) düşüşü 1990'a yaklaşırken
4. **ASEAN evolution** (1967-2016) — 5'ten 10'a
5. **BRICS** (2009-2016) — test döneminde aktif koalisyon

**Beklenen bulgular:**
- Tarihsel olaylar (genişleme yılları, çöküşler) v(S) trajectory'sinde **görünür dönüm noktaları** üretmeli
- Test dönemindeki (2010-2016) koalisyonlar modelin **gerçek tahmin yeteneğini** gösterir

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/coalition_gnn'
SNAP_DIR = os.path.join(PROJECT_DIR, 'data/snapshots')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
CANON_DIR = os.path.join(PROJECT_DIR, 'data/canonical')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
import matplotlib.pyplot as plt

EDGE_TYPES = ['allies', 'trades', 'votes_with', 'conflicts_with']
NUM_RELATIONS = 4

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Snapshot'ları yükle (heterojen, 07 ile aynı)
def load_snapshot(year):
    path = os.path.join(SNAP_DIR, f'graph_{year}.pt')
    if not os.path.exists(path):
        return None
    data = torch.load(path, weights_only=False)
    x = data['country'].x
    cow_codes = data['country'].cow_code.numpy()
    
    ei_list, et_list = [], []
    for rel_idx, rel_name in enumerate(EDGE_TYPES):
        key = ('country', rel_name, 'country')
        if key in data.edge_types:
            ei = data[key].edge_index
            ei_list.append(ei)
            et_list.append(torch.full((ei.shape[1],), rel_idx, dtype=torch.long))
    
    if not ei_list:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_type = torch.zeros(0, dtype=torch.long)
    else:
        edge_index = torch.cat(ei_list, dim=1)
        edge_type = torch.cat(et_list)
    
    return {
        'x': x.to(DEVICE),
        'edge_index': edge_index.to(DEVICE),
        'edge_type': edge_type.to(DEVICE),
        'cow_codes': cow_codes,
        'code_to_idx': {int(c): i for i, c in enumerate(cow_codes)},
        'year': year,
    }

snapshots = {}
for year in range(1946, 2017):
    s = load_snapshot(year)
    if s: snapshots[year] = s

NUM_FEATURES = next(iter(snapshots.values()))['x'].shape[1]
print(f'Yüklendi: {len(snapshots)} yıl, {NUM_FEATURES} özellik')

In [ ]:
# 07 ile aynı model mimarisi
class RGCNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, dropout=0.3):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = RGCNConv(hidden_channels, out_channels, num_relations)
        self.dropout = dropout
        self.norm1 = nn.LayerNorm(hidden_channels)
        self.norm2 = nn.LayerNorm(out_channels)
    def forward(self, x, edge_index, edge_type):
        h = self.conv1(x, edge_index, edge_type); h = self.norm1(h); h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, edge_index, edge_type); h = self.norm2(h)
        return h

class RichCoalitionHead(nn.Module):
    def __init__(self, embed_dim, raw_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        pool_dim = 4 * (embed_dim + raw_dim) + 2
        self.W_pair = nn.Parameter(torch.randn(embed_dim, embed_dim) * 0.01)
        self.validity_mlp = nn.Sequential(
            nn.Linear(pool_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
        embed_pool_dim = 4 * embed_dim + 1
        self.duration_head = nn.Sequential(
            nn.Linear(embed_pool_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1), nn.Softplus(),
        )
        self.cohesion_head = nn.Sequential(
            nn.Linear(embed_pool_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1), nn.Sigmoid(),
        )
    def rich_pool(self, feats):
        std = feats.std(dim=0) if feats.shape[0] > 1 else torch.zeros_like(feats[0])
        return torch.cat([feats.mean(dim=0), std, feats.max(dim=0).values, feats.min(dim=0).values])
    def forward(self, z, x_raw):
        k = z.shape[0]
        size_feat = torch.tensor([k / 20.0], device=z.device)
        z_pool = self.rich_pool(z); raw_pool = self.rich_pool(x_raw)
        if k >= 2:
            Wz = z @ self.W_pair; pm = z @ Wz.T
            pair_score = (pm.sum() - pm.diag().sum()) / (k * (k - 1))
        else:
            pair_score = torch.tensor(0.0, device=z.device)
        v_input = torch.cat([z_pool, raw_pool, size_feat, pair_score.unsqueeze(0)])
        validity = self.validity_mlp(v_input).squeeze()
        mt_input = torch.cat([z_pool, size_feat])
        duration = self.duration_head(mt_input).squeeze()
        cohesion = self.cohesion_head(mt_input).squeeze()
        return validity, duration, cohesion

class FullRichGNN(nn.Module):
    def __init__(self, num_features, hidden_dim=128, embed_dim=128, num_relations=4, head_hidden=128, dropout=0.3):
        super().__init__()
        self.encoder = RGCNEncoder(num_features, hidden_dim, embed_dim, num_relations, dropout)
        self.head = RichCoalitionHead(embed_dim, num_features, head_hidden, dropout)
    def forward_sample(self, snap, member_idx_list):
        z_all = self.encoder(snap['x'], snap['edge_index'], snap['edge_type'])
        idx = torch.tensor(member_idx_list, dtype=torch.long, device=DEVICE)
        return self.head(z_all[idx], snap['x'][idx])

# Modeli yükle
model = FullRichGNN(NUM_FEATURES).to(DEVICE)
ckpt_path = os.path.join(MODELS_DIR, 'full_rich_gnn.pt')
model.load_state_dict(torch.load(ckpt_path, weights_only=False))
model.eval()
print(f'✓ Model yüklendi: {ckpt_path}')

In [ ]:
# COW code → ülke ismi mapping
country_master = pd.read_parquet(os.path.join(CANON_DIR, 'country_master.parquet'))
cow_to_name = {}
for _, row in country_master.iterrows():
    cow_to_name[int(row['cow_code'])] = row.get('state_name', f"COW{row['cow_code']}")

# v(S) trajectory hesapla — bir koalisyonu yıl yıl skorla
def compute_v_trajectory(members_cow, year_range):
    """Verilen üyeler için her yıl v(S) (sigmoid logit) hesapla."""
    trajectory = {}
    with torch.no_grad():
        for year in year_range:
            if year not in snapshots:
                continue
            snap = snapshots[year]
            valid = [m for m in members_cow if m in snap['code_to_idx']]
            if len(valid) < 2:
                continue
            idx = [snap['code_to_idx'][m] for m in valid]
            v_logit, _, _ = model.forward_sample(snap, idx)
            trajectory[year] = {
                'v_prob': torch.sigmoid(v_logit).item(),
                'v_logit': v_logit.item(),
                'n_members': len(valid),
            }
    return trajectory

def plot_trajectory(ax, traj, label, color='steelblue', events=None):
    years = sorted(traj.keys())
    probs = [traj[y]['v_prob'] for y in years]
    ax.plot(years, probs, marker='o', markersize=4, color=color, label=label, linewidth=2)
    if events:
        for ev_year, ev_label in events:
            ax.axvline(ev_year, ls='--', color='red', alpha=0.4)
            ax.text(ev_year, 0.02, ev_label, rotation=90, fontsize=8,
                    verticalalignment='bottom', alpha=0.7)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel('v(S) probability')
    ax.grid(alpha=0.3)
    ax.legend()

print('✓ Helper fonksiyonlar hazır')

## Case Study 1: NATO Evolution (1949-2016)

NATO 7 ayrı üyelik versiyonu yaşadı. Her versiyondaki üye seti ile v(S) trajectory'sini kendi dönemini kapsayacak şekilde çizeriz.

In [ ]:
# NATO versiyonları (coalitions_canon.yaml'dan)
nato_versions = [
    ('NATO_founding', 'NATO-12 (1949-1951)', [2,20,200,210,211,212,220,235,325,385,390,395]),
    ('NATO_15',       'NATO-14 (1952-54: +GRC,TUR)', [2,20,200,210,211,212,220,235,325,350,385,390,395,640]),
    ('NATO_16_GFR',   'NATO-15 (1955-81: +GFR)',     [2,20,200,210,211,212,220,235,260,325,350,385,390,395,640]),
    ('NATO_16_SPN',   'NATO-16 (1982-98: +SPN)',     [2,20,200,210,211,212,220,230,235,255,325,350,385,390,395,640]),
    ('NATO_1999',     'NATO-19 (1999-03: +POL,HUN,CZR)', [2,20,200,210,211,212,220,230,235,255,290,310,316,325,350,385,390,395,640]),
    ('NATO_2004',     'NATO-26 (2004+: +7 ülke)',    [2,20,200,210,211,212,220,230,235,255,290,310,316,317,325,349,350,355,360,366,367,368,385,390,395,640]),
]

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(nato_versions)))

for (vid, vname, members), color in zip(nato_versions, colors):
    traj = compute_v_trajectory(members, range(1946, 2017))
    plot_trajectory(ax, traj, vname, color=color)

events = [(1949, 'NATO kuruldu'), (1955, 'Varşova Paktı'), (1991, 'SSCB çöküşü'),
          (1999, 'Doğu genişlemesi'), (2004, 'Büyük genişleme')]
for ev_year, ev_label in events:
    ax.axvline(ev_year, ls='--', color='red', alpha=0.3)
    ax.text(ev_year+0.2, 0.05, ev_label, rotation=90, fontsize=8, alpha=0.6)

ax.set_title('NATO Evolution — v(S) trajectory (her versiyonun üye seti, tüm yıllarda)')
ax.set_xlabel('Yıl')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'case_nato.png'), dpi=120)
plt.show()

## Case Study 2: AB Evolution (1957-2016)

In [ ]:
eu_versions = [
    ('EU_6',  'AET-6 (1957-72)',   [210,211,212,220,260,325]),
    ('EU_9',  'AET-9 (1973-80)',   [200,205,210,211,212,220,260,325,390]),
    ('EU_12', 'AT-12 (1986-94)',   [200,205,210,211,212,220,230,235,255,260,325,350,390]),
    ('EU_15', 'AB-15 (1995-03)',   [200,205,210,211,212,220,230,235,255,305,325,350,375,380,390]),
    ('EU_25', 'AB-25 (2004-06)',   [200,205,210,211,212,220,230,235,255,290,305,310,316,317,325,338,349,350,352,366,367,368,375,380,390]),
    ('EU_27', 'AB-27 (2007-12)',   [200,205,210,211,212,220,230,235,255,290,305,310,316,317,325,338,349,350,352,355,360,366,367,368,375,380,390]),
    ('EU_28', 'AB-28 (2013+)',     [200,205,210,211,212,220,230,235,255,290,305,310,316,317,325,338,344,349,350,352,355,360,366,367,368,375,380,390]),
]

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.plasma(np.linspace(0, 0.85, len(eu_versions)))

for (vid, vname, members), color in zip(eu_versions, colors):
    traj = compute_v_trajectory(members, range(1946, 2017))
    plot_trajectory(ax, traj, vname, color=color)

events = [(1957, 'Roma Anlş.'), (1992, 'Maastricht'), (2004, 'Büyük genişleme'),
          (2008, 'Mali kriz'), (2013, 'Hırvatistan'), (2016, 'Brexit ref.')]
for ev_year, ev_label in events:
    ax.axvline(ev_year, ls='--', color='red', alpha=0.3)
    ax.text(ev_year+0.2, 0.05, ev_label, rotation=90, fontsize=8, alpha=0.6)

ax.set_title('Avrupa Birliği Evolution — v(S) trajectory')
ax.set_xlabel('Yıl')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'case_eu.png'), dpi=120)
plt.show()

## Case Study 3: Varşova Paktı Çöküşü (1955-1991)

In [ ]:
warsaw_members = [265, 290, 310, 315, 339, 355, 360, 365]  # GDR,POL,HUN,CZE,ALB,BUL,ROM,USSR

traj_warsaw = compute_v_trajectory(warsaw_members, range(1955, 1995))

fig, ax = plt.subplots(figsize=(12, 5))
plot_trajectory(ax, traj_warsaw, 'Varşova Paktı', color='darkred')

events = [(1955, 'Kuruluş'), (1956, 'Macaristan'), (1968, 'Prag Baharı'),
          (1980, 'Solidarity'), (1989, 'Berlin Duvarı'), (1991, 'Çöküş')]
for ev_year, ev_label in events:
    ax.axvline(ev_year, ls='--', color='steelblue', alpha=0.4)
    ax.text(ev_year+0.2, 0.05, ev_label, rotation=90, fontsize=8, alpha=0.7)

ax.set_title('Varşova Paktı çöküşüne giden v(S) düşüşü')
ax.set_xlabel('Yıl')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'case_warsaw.png'), dpi=120)
plt.show()

# v(S) maks ve son değer arasındaki düşüşü ölç
years_sorted = sorted(traj_warsaw.keys())
probs = [traj_warsaw[y]['v_prob'] for y in years_sorted]
peak_year = years_sorted[np.argmax(probs)]
print(f'Zirve v(S): yıl {peak_year} → p={max(probs):.4f}')
print(f'1989 v(S): p={traj_warsaw.get(1989, {}).get("v_prob", float("nan")):.4f}')
print(f'1991 v(S): p={traj_warsaw.get(1991, {}).get("v_prob", float("nan")):.4f}')

## Case Study 4: ASEAN Evolution (1967-2016)

In [ ]:
asean_versions = [
    ('ASEAN_founding', 'ASEAN-5 (1967-83)', [800,820,830,840,850]),
    ('ASEAN_expanded', 'ASEAN-6 (1984-94)', [800,820,830,835,840,850]),
    ('ASEAN_10',       'ASEAN-10 (1999+)',  [775,800,811,812,816,820,830,835,840,850]),
]

fig, ax = plt.subplots(figsize=(13, 5))
colors = ['steelblue', 'orange', 'seagreen']

for (vid, vname, members), color in zip(asean_versions, colors):
    traj = compute_v_trajectory(members, range(1946, 2017))
    plot_trajectory(ax, traj, vname, color=color)

events = [(1967, 'Bangkok'), (1984, 'Brunei'), (1995, 'Vietnam'), (1997, 'Laos+Myanmar'), (1999, 'Kamboçya')]
for ev_year, ev_label in events:
    ax.axvline(ev_year, ls='--', color='red', alpha=0.3)
    ax.text(ev_year+0.2, 0.05, ev_label, rotation=90, fontsize=8, alpha=0.6)

ax.set_title('ASEAN Evolution — v(S) trajectory')
ax.set_xlabel('Yıl')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'case_asean.png'), dpi=120)
plt.show()

## Case Study 5: BRICS (2009-2016) — Test Dönemi

Bu koalisyon **test setinde aktif** (2010-2016). Modelin tahmin gücünü en doğrudan test eder.

In [ ]:
brics_members = [140, 365, 560, 710, 750]  # BRA, RUS, SAF, CHN, IND

# BRICS öncesi yılları da göster (placebo): koalisyon olmadığı dönemde
traj_brics = compute_v_trajectory(brics_members, range(1990, 2017))

fig, ax = plt.subplots(figsize=(12, 5))
plot_trajectory(ax, traj_brics, 'BRICS', color='purple')

ax.axvspan(2009, 2016, alpha=0.10, color='green', label='Resmi BRICS dönemi')
ax.axvline(2010, ls='--', color='blue', alpha=0.4)
ax.text(2010.2, 0.05, 'SAF katıldı', rotation=90, fontsize=8, alpha=0.7)
ax.axvline(2014, ls='--', color='blue', alpha=0.4)
ax.text(2014.2, 0.05, 'Kırım/Yaptırımlar', rotation=90, fontsize=8, alpha=0.7)

ax.set_title('BRICS — Test döneminde modelin v(S) tahmini')
ax.set_xlabel('Yıl')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'case_brics.png'), dpi=120)
plt.show()

print('BRICS yılları:')
for y in sorted(traj_brics.keys()):
    if y >= 2005:
        print(f"  {y}: v(S)={traj_brics[y]['v_prob']:.4f}")

## Özet — Tarihsel Doğrulama Sonuçları

Tüm case study'lerin v(S) trajectory'lerini tek bir panelde topla.

In [ ]:
summary = {
    'NATO-26 (post-2004)': nato_versions[5][2],
    'AB-28 (post-2013)':   eu_versions[6][2],
    'Varşova Paktı':       warsaw_members,
    'ASEAN-10 (post-1999)': asean_versions[2][2],
    'BRICS':               brics_members,
}

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.tab10(range(len(summary)))

for (name, members), color in zip(summary.items(), colors):
    traj = compute_v_trajectory(members, range(1946, 2017))
    plot_trajectory(ax, traj, name, color=color)

# Test dönemini vurgula
ax.axvspan(2010, 2016, alpha=0.10, color='gray', label='Test dönemi (2010-2016)')
ax.set_title('Özet: 5 büyük koalisyonun v(S) trajectory\'si')
ax.set_xlabel('Yıl')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'case_summary.png'), dpi=120)
plt.show()

## Makale için yorumlanacak bulgular

Bu hücreyi çalıştırdıktan sonra şu soruları cevaplayacağız:

1. **NATO trajectory**: Her genişleme sonrası v(S) ne yapıyor? Soğuk Savaş bitişinde (1991) düşüyor mu?
2. **AB trajectory**: Maastricht (1992), Doğu genişleme (2004), Brexit ref. (2016) v(S)'de görünüyor mu?
3. **Varşova Paktı**: v(S) gerçekten 1989-1991 arasında çöküşü öngörüyor mu?
4. **ASEAN**: Genişleme dönemlerinde v(S) artıyor mu yoksa düşüyor mu?
5. **BRICS**: 2009 öncesinde (olmadığı dönem) düşük, sonrasında yüksek v(S) gösteriyor mu?

Bu bulgular doğrudan makalenin **§5 Experiments — Case Studies** bölümüne girecek.

## 📋 Paylaşılabilir Metin Özeti

Bu hücreyi çalıştır, çıktıyı Claude'a kopyala-yapıştır.

In [ ]:
# === 08_CASE_STUDIES — PAYLAŞIM ÖZETİ ===
def format_traj_summary(traj, name, key_years=None):
    if not traj:
        return f"  {name}: (veri yok)"
    years = sorted(traj.keys())
    probs = [traj[y]['v_prob'] for y in years]
    peak_idx = int(np.argmax(probs))
    trough_idx = int(np.argmin(probs))
    lines = [f"\n--- {name} ---"]
    lines.append(f"  Yıl aralığı: {years[0]}-{years[-1]} ({len(years)} yıl)")
    lines.append(f"  Peak  v(S): {probs[peak_idx]:.4f} @ {years[peak_idx]}")
    lines.append(f"  Trough v(S): {probs[trough_idx]:.4f} @ {years[trough_idx]}")
    lines.append(f"  Ortalama:   {np.mean(probs):.4f}  (std {np.std(probs):.4f})")
    if key_years:
        lines.append("  Önemli yıllar:")
        for ky in key_years:
            if ky in traj:
                lines.append(f"    {ky}: v(S)={traj[ky]['v_prob']:.4f}")
    return '\n'.join(lines)

print('='*70)
print('CASE STUDY ÖZETİ — Claude ile paylaşmak için kopyala-yapıştır')
print('='*70)

# NATO versiyonları
print('\n### NATO ###')
for vid, vname, members in nato_versions:
    traj = compute_v_trajectory(members, range(1946, 2017))
    print(format_traj_summary(traj, vname,
        key_years=[1949, 1955, 1989, 1991, 1999, 2004, 2016]))

# AB versiyonları
print('\n\n### AVRUPA BİRLİĞİ ###')
for vid, vname, members in eu_versions:
    traj = compute_v_trajectory(members, range(1946, 2017))
    print(format_traj_summary(traj, vname,
        key_years=[1957, 1973, 1992, 2004, 2008, 2013, 2016]))

# Varşova Paktı
print('\n\n### VARŞOVA PAKTI ###')
warsaw_traj = compute_v_trajectory(warsaw_members, range(1955, 1995))
print(format_traj_summary(warsaw_traj, 'Varşova Paktı',
    key_years=[1955, 1968, 1980, 1985, 1989, 1990, 1991]))

# ASEAN
print('\n\n### ASEAN ###')
for vid, vname, members in asean_versions:
    traj = compute_v_trajectory(members, range(1946, 2017))
    print(format_traj_summary(traj, vname,
        key_years=[1967, 1984, 1995, 1999, 2010, 2016]))

# BRICS
print('\n\n### BRICS (test dönemi) ###')
brics_traj = compute_v_trajectory(brics_members, range(1990, 2017))
print(format_traj_summary(brics_traj, 'BRICS',
    key_years=[1990, 2000, 2008, 2009, 2010, 2014, 2016]))

print('\n' + '='*70)
print('Yıl yıl tüm değerler tablolaştırılmış (Markdown):')
print('='*70)

# Tek tablo: tüm case study'lerin v(S) değerleri
all_case_trajs = {
    'NATO-26':       compute_v_trajectory(nato_versions[5][2], range(1946, 2017)),
    'AB-28':         compute_v_trajectory(eu_versions[6][2],   range(1946, 2017)),
    'Varşova Paktı': warsaw_traj,
    'ASEAN-10':      compute_v_trajectory(asean_versions[2][2], range(1946, 2017)),
    'BRICS':         brics_traj,
}

# 5 yıllık adım, paylaşılabilir kompakt tablo
print('\n| Yıl | NATO-26 | AB-28 | Varşova | ASEAN-10 | BRICS |')
print('|-----|---------|-------|---------|----------|-------|')
for year in range(1950, 2017, 5):
    row = [str(year)]
    for cname in ['NATO-26', 'AB-28', 'Varşova Paktı', 'ASEAN-10', 'BRICS']:
        v = all_case_trajs[cname].get(year, {}).get('v_prob')
        row.append(f"{v:.3f}" if v is not None else '-')
    print('| ' + ' | '.join(row) + ' |')

# Test dönemini detaylı (2010-2016 her yıl)
print('\n### Test dönemi detayı (2010-2016) ###')
print('| Yıl | NATO-26 | AB-28 | ASEAN-10 | BRICS |')
print('|-----|---------|-------|----------|-------|')
for year in range(2010, 2017):
    row = [str(year)]
    for cname in ['NATO-26', 'AB-28', 'ASEAN-10', 'BRICS']:
        v = all_case_trajs[cname].get(year, {}).get('v_prob')
        row.append(f"{v:.4f}" if v is not None else '-')
    print('| ' + ' | '.join(row) + ' |')
